# v18 — pending T4 experiment

GPU tests, speed, quality and ten rule checks have **not yet been validated**.
v16 remains the verified fallback. This notebook pins the exact experimental source.
Use GPU T4 x2, Internet On, and Run All. Each paired run uses one idle GPU.


In [ ]:
REPO_URL = "https://github.com/promaladoi/Decentrathon-6.0.git"
COMMIT = "d01b2d8c71738c6345cd704b369bd58027ac09c1"
VERSION = "v18"


In [ ]:
import subprocess, os, json, zipfile, uuid
from pathlib import Path
from IPython.display import display, FileLink
print(subprocess.check_output(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],text=True))
subprocess.run(['python','-m','pip','install','-q','soup-cli[train]==0.77.0','transformers==5.16.1','peft==0.20.0','trl==0.29.1','bitsandbytes==0.50.2','accelerate==1.14.0','pytest'],check=True)


In [ ]:
assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip(), 'GPU is busy: stop the other GPU work before running this notebook.'
p=Path('/kaggle/working')/('nurzhigit_'+VERSION+'_'+uuid.uuid4().hex[:8])
subprocess.run(['git','clone','-q',REPO_URL,str(p)],check=True,timeout=120)
subprocess.run(['git','-C',str(p),'checkout','-q',COMMIT],check=True)
print('MEASURED CODE:',subprocess.check_output(['git','-C',str(p),'rev-parse','HEAD'],text=True).strip())
subprocess.run(['python','task/prepare_data.py'],cwd=p,check=True)


In [ ]:
completed = subprocess.run(['python','04_submission/run_candidate.py','--version',VERSION],cwd=p)
display(FileLink('codex_'+VERSION+'.zip'))
assert completed.returncode == 0, 'Experiment stopped or rejected; inspect its output/archive.'
